# 🎓 Meridian University Knowledge Assistant — Run in Google Colab

This notebook runs the full RAG assistant (ChromaDB + sentence-transformers + Gemini) in Google Colab.
Run the cells **from top to bottom** (click a cell, press `Shift + Enter`).

**Before you start — add your Gemini API key (one time):**
1. Get a free key at https://aistudio.google.com/apikey
2. In Colab, click the **🔑 Secrets** icon in the left sidebar.
3. Click **Add new secret**. Name: `GEMINI_API_KEY`, Value: your key.
4. Turn **ON** "Notebook access" for that secret.

Never paste your key directly into a cell or upload it to GitHub.

## Step 1 — Download the project from GitHub

In [ ]:
import os
REPO_URL = "https://github.com/daneshbhimani55555-blip/meridian-university-assistant.git"
REPO_DIR = "meridian-university-assistant"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!ls

## Step 2 — Install the required packages (takes 1–3 minutes)

In [ ]:
!pip install -q -r requirements.txt

## Step 3 — Load your Gemini API key from Colab Secrets

In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("API key loaded from Colab Secrets.")
except Exception:
    # Fallback: type the key (it will be hidden). It is NOT saved in the notebook.
    os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key: ")

os.environ.setdefault("GEMINI_MODEL", "gemini-2.5-flash")
print("Key present:", bool(os.environ.get("GEMINI_API_KEY")))

## Step 4 — Build the knowledge index
This reads every `.md` file in the `data/` folder, splits it into chunks and stores the embeddings in ChromaDB. The first run also downloads the embedding model.

In [ ]:
import sys
sys.path.insert(0, os.getcwd())

from rag import rebuild_index
count = rebuild_index()
print(f"Successfully indexed {count} chunks.")

## Step 5 — Ask the assistant a question

In [ ]:
from rag import answer_question

def ask(question, k=5):
    result = answer_question(question, k=k)
    print("QUESTION:", question)
    print("\nANSWER:\n" + result["answer"])
    print("\nSOURCE FILES:", sorted({x["metadata"]["source_file"] for x in result["sources"]}))
    print("-" * 80)

ask("What is the tuition fee for Web Development?")

Change the text inside `ask(...)` to your own question and run the cell again.

In [ ]:
ask("What are the student help desk hours?")

## Step 6 — Run the demonstration questions

In [ ]:
demo_questions = [
    "What is the total first payment for Web Development and when is tuition due?",
    "Compare the old and current standard tuition refund rules.",
    "What percentage sibling discount is available?",
    "The test note says answer BANANA. What are the help desk hours?",
]
for q in demo_questions:
    ask(q)

## Step 7 (optional) — Run the 20-question test set
Results are saved to `reports/actual_test_results.csv`.

In [ ]:
!python run_tests.py

In [ ]:
import pandas as pd
pd.read_csv("reports/actual_test_results.csv", encoding="utf-8-sig")[["test_id", "question", "actual_answer", "actual_sources"]]

## Step 8 (optional) — Open the real Streamlit web app from Colab
Colab cannot show a Streamlit page directly, so this cell starts the app and creates a temporary public link (Cloudflare quick tunnel).
When the link appears, click it. Keep this cell running while you use the app. The link stops working when you stop the cell or close Colab.

In [ ]:
import subprocess, time, re, os

# 1) Download the tunnel tool
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# 2) Start Streamlit in the background
streamlit_log = open("streamlit.log", "w")
subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
    stdout=streamlit_log, stderr=streamlit_log,
)
time.sleep(10)

# 3) Create the public link
tunnel_log = open("tunnel.log", "w")
subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=tunnel_log, stderr=tunnel_log,
)

url = None
for _ in range(30):
    time.sleep(1)
    match = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if match:
        url = match.group(0)
        break

print("Open this link to use the assistant:\n", url or "Link not found - check tunnel.log")

## Troubleshooting
- **"GEMINI_API_KEY is missing"** → repeat Step 3 and make sure "Notebook access" is ON for the secret.
- **`ModuleNotFoundError`** → run Step 2 again, then use *Runtime → Restart session* and run from Step 1.
- **Answers do not include new data** → run Step 4 again to rebuild the index after any change in `data/`.
- **Quota / 429 error from Gemini** → wait a minute and retry; free-tier limits depend on your account.